In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

### conf

In [ ]:
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf, open_dict

cfg_root = "/home/n.sorokin/verl-agent/verl/trainer/config"
with initialize_config_dir(config_dir=cfg_root, version_base=None):
    cfg = compose(
        config_name="ppo_trainer",
        overrides=[
            "data.train_files=/home/n.sorokin/data/verl-agent/text/train.parquet",
            "data.val_files=/home/n.sorokin/data/verl-agent/text/test.parquet",
            "actor_rollout_ref.model.path=Qwen/Qwen2.5-1.5B-Instruct",
            "env.env_name=craftext/CraftextEnv",
            "actor_rollout_ref.rollout.name=vllm",
        ],
    )

with open_dict(cfg):
    cfg.trainer.resume_mode = "resume_path"
    cfg.trainer.resume_from_path = "/home/n.sorokin/verl-agent/checkpoints/verl_agent_craftext/gigpo_qwen2.5_1.5b_lora_run_20251110-160709/global_step_500"
    cfg.trainer.val_only = True             # stop after one validation pass
    cfg.trainer.val_before_train = True     # ensure validation runs
    cfg.trainer.num_samples_to_log = 1
    cfg.trainer.logger = ["console"]
    cfg.data.val_batch_size = 1
    cfg.data.train_batch_size = 1
    cfg.env.rollout.n = 1
    cfg.actor_rollout_ref.rollout.val_kwargs.n = 1
    cfg.actor_rollout_ref.rollout.val_kwargs.do_sample = False  # deterministic
    cfg.actor_rollout_ref.actor.micro_batch_size = 1
    cfg.actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu = 1
    cfg.actor_rollout_ref.rollout.log_prob_micro_batch_size_per_gpu = 1
    cfg.actor_rollout_ref.ref.log_prob_micro_batch_size_per_gpu = 1
    cfg.algorithm.adv_estimator = "grpo"  # optional: skips the critic
    cfg.actor_rollout_ref.rollout.tensor_model_parallel_size = 1
    cfg.trainer.n_gpus_per_node = 1
    cfg.trainer.nnodes = 1
    cfg.env.craftext_settings = "achievements_wood"
    cfg.actor_rollout_ref.model.lora_rank = 64
    cfg.actor_rollout_ref.model.lora_alpha = 64
    cfg.actor_rollout_ref.model.target_modules = "all-linear"
    cfg.actor_rollout_ref.model.use_remove_padding = True
    cfg.actor_rollout_ref.actor.optim.lr = 1e-6
    cfg.actor_rollout_ref.actor.ppo_mini_batch_size = 16
    cfg.actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu = 8
    cfg.actor_rollout_ref.actor.ppo_mini_batch_size = 16
    cfg.data.return_raw_chat = True
    cfg.data.max_prompt_length = 2048
    cfg.data.max_response_length = 512

In [ ]:
import ray, torch
from verl.utils import hf_tokenizer, hf_processor
from agent_system.environments import make_envs
from agent_system.multi_turn_rollout import TrajectoryCollector
from verl.trainer.ppo.ray_trainer import RayPPOTrainer, ResourcePoolManager, Role
from verl.single_controller.ray import RayWorkerGroup
from verl.workers.fsdp_workers import ActorRolloutRefWorker, CriticWorker

ray.init(num_cpus=cfg.ray_init.num_cpus)

# HF artifacts for the base model
tokenizer = hf_tokenizer(cfg.actor_rollout_ref.model.path, trust_remote_code=cfg.data.get("trust_remote_code", False))
processor = hf_processor(cfg.actor_rollout_ref.model.path, trust_remote_code=cfg.data.get("trust_remote_code", False), use_fast=True)

# Environments (train + val); you only need val_envs for exploration
envs, val_envs = make_envs(cfg)

traj_collector = TrajectoryCollector(config=cfg, tokenizer=tokenizer, processor=processor)

In [ ]:
from verl.trainer.ppo.ray_trainer import Role, RayClassWithInitArgs

role_worker_mapping = {
    Role.ActorRollout: ray.remote(ActorRolloutRefWorker),
    Role.Critic: ray.remote(CriticWorker),
}
resource_pool_spec = {"pool": [cfg.trainer.n_gpus_per_node] * cfg.trainer.nnodes}
resource_pool_manager = ResourcePoolManager(resource_pool_spec=resource_pool_spec, mapping={
    Role.ActorRollout: "pool",
    Role.Critic: "pool",
})

trainer = RayPPOTrainer(
    config=cfg,
    tokenizer=tokenizer,
    processor=processor,
    role_worker_mapping=role_worker_mapping,
    resource_pool_manager=resource_pool_manager,
    ray_worker_group_cls=RayWorkerGroup,
    reward_fn=None,              # rule-based EpisodeRewardManager is instantiated inside _validate
    val_reward_fn=None,
    traj_collector=traj_collector,
    envs=envs,
    val_envs=val_envs,
)
trainer.init_workers()
trainer._load_checkpoint()

In [ ]:
from verl import DataProto
from agent_system.multi_turn_rollout.utils import to_list_of_dict

val_iter = iter(trainer.val_dataloader)
batch_dict = next(val_iter)                                   # only 1 sample
test_batch = DataProto.from_single_dict(batch_dict)

batch_keys = ["input_ids", "attention_mask", "position_ids"]
non_tensor_keys = ["raw_prompt_ids"]
test_gen_batch = test_batch.pop(batch_keys=batch_keys, non_tensor_batch_keys=non_tensor_keys)
test_gen_batch.meta_info.update({
    "do_sample": cfg.actor_rollout_ref.rollout.val_kwargs.do_sample,
    "temperature": cfg.actor_rollout_ref.rollout.val_kwargs.temperature,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.pad_token_id,
})

if "raw_prompt" not in test_gen_batch.non_tensor_batch:
    decoded = tokenizer.batch_decode(
        test_gen_batch.non_tensor_batch["raw_prompt_ids"],
        skip_special_tokens=False,
    )
    test_gen_batch.non_tensor_batch["raw_prompt"] = [
        [{"role": "user", "content": text}] for text in decoded
    ]

if "data_source" not in test_gen_batch.non_tensor_batch:
    test_gen_batch.non_tensor_batch["data_source"] = ["craftext"] * len(test_gen_batch)

rollout = trainer.traj_collector.multi_turn_loop(
    gen_batch=test_gen_batch,
    actor_rollout_wg=trainer.actor_rollout_wg,
    envs=trainer.val_envs,
    is_train=False,
)
trajectory = to_list_of_dict(rollout)[0]     # first/only env in the batch

In [ ]:
traj = to_list_of_dict(rollout)

In [ ]:
print(len(traj))
print(traj[0].keys()) 

In [ ]:
from agent_system.environments.env_package.craftext.projection import craftext_projection

action_ids =traj[0]['responses']
print("Action token IDs:", action_ids)

text_actions = tokenizer.batch_decode(action_ids, skip_special_tokens=True)
print("Decoded text actions:", text_actions)

# Проверяем валидность действий через projection
action_ids_parsed, valids = craftext_projection(text_actions)
print(f"Parsed action IDs: {action_ids_parsed}")
print(f"Action validities: {valids}")
print(f"is_action_valid from trajectory: {traj[0]['is_action_valid']}")

# Проверяем награды
print(f"\nRewards: {traj[0]['rewards']}")
print(f"Total reward: {sum(traj[0]['rewards']):.4f}")
print(f"Episode reward (last step): {traj[0]['rewards'][-1] if len(traj[0]['rewards']) > 0 else 0:.4f}")

# Проверяем первые несколько шагов детально
print("\n" + "="*50)
print("Первые 3 шага детально:")
for i in range(min(3, len(text_actions))):
    print(f"\nШаг {i}:")
    print(f"  Текст действия: {text_actions[i]}")
    print(f"  Parsed action ID: {action_ids_parsed[i]}")
    print(f"  Valid: {bool(valids[i])}")
    print(f"  Reward: {traj[0]['rewards'][i]:.4f}")
    print(f"  Done: {not bool(traj[0]['active_masks'][i])}")
# print(traj[0]['prompts'])
# print(traj[0]['responses'])
# print(traj[0]['rewards'])
# print(traj[0]['active_masks'])
# print(traj[0]['anchor_obs'])


In [ ]:
summary = {}
for key, value in trajectory.items():
    if isinstance(value, (list, tuple)):
        summary[key] = (type(value[0]).__name__, len(value))
    else:
        summary[key] = (type(value).__name__, None)
summary


In [ ]:
import pandas as pd

step_count = len(trajectory["responses"])
print(step_count)

records = []
for step in range(step_count):
    obs_text = trajectory["anchor_obs"][step]
    action_ids = trajectory["responses"][step]
    action_ids = action_ids.tolist() if hasattr(action_ids, "tolist") else action_ids
    action_text = tokenizer.decode(action_ids, skip_special_tokens=True).strip()
    reward = float(trajectory["rewards"][step])
    done = not bool(trajectory["active_masks"][step])
    records.append({
        "step": step,
        "observation": obs_text,
        "action": action_text,
        "reward": reward,
        "done": done,
    })
traj_df = pd.DataFrame(records)
traj_df.head()


In [ ]:
step = 0
print("Observation:\n", traj_df.loc[step, "observation"])
print("\nAction:\n", traj_df.loc[step, "action"])
print("\nReward:", traj_df.loc[step, "reward"], " Done:", traj_df.loc[step, "done"])


In [ ]:
trainer.actor_rollout_wg.shutdown()
if trainer.use_critic:
    trainer.critic_wg.shutdown()
trainer.resource_pool_manager.destroy()
ray.shutdown()


### Простой интерфейс Agent для инференса

Простой класс Agent с gym-подобным интерфейсом для инференса модели без Ray workers.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# SimpleAgent: простой класс для инференса модели с gym-подобным интерфейсом

import os
import multiprocessing

# Устанавливаем переменную окружения ДО импорта vLLM
# Это заставляет vLLM использовать старую версию движка, которая лучше работает в Jupyter
os.environ.setdefault('VLLM_USE_V1', '0')

from transformers import AutoConfig
from verl.utils import hf_tokenizer, hf_processor
from vllm import LLM, SamplingParams

# Устанавливаем метод запуска multiprocessing на 'spawn' для работы с CUDA
# Это необходимо для vLLM в Jupyter notebooks
try:
    multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
    # Метод уже установлен, это нормально
    pass

/home/n.sorokin/miniconda3/envs/verl-agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-26 15:26:54,888	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-26 15:26:58 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 11-26 15:27:02 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
from simple_agent import SimpleAgent

In [3]:
# Пример использования SimpleAgent

# Создаем Agent
checkpoint_path = "/home/n.sorokin/verl-agent/checkpoints/verl_agent_craftext/gigpo_qwen2.5_1.5b_lora_run_20251110-160709/global_step_500"
# checkpoint_path = ""

# Проверяем наличие чекпоинта и LoRA адаптера
import os
print(f"Проверка чекпоинта: {checkpoint_path}")
print(f"  Чекпоинт существует: {os.path.exists(checkpoint_path)}")

lora_path = os.path.join(checkpoint_path, "actor", "lora_adapter")
print(f"  LoRA путь: {lora_path}")
print(f"  LoRA существует: {os.path.exists(lora_path)}")

if os.path.exists(lora_path):
    adapter_files = os.listdir(lora_path)
    print(f"  LoRA файлы: {adapter_files}")

agent = SimpleAgent(
    checkpoint_path=checkpoint_path,
    base_model_path="Qwen/Qwen2.5-1.5B-Instruct",
    lora_rank=64,
    lora_alpha=64,
    target_modules="all-linear",
    max_response_length=512,
    do_sample=True,  # ИСПРАВЛЕНО: должно совпадать с обучением (val_kwargs.do_sample=True)
    temperature=0.4,  # ИСПРАВЛЕНО: должно совпадать с обучением (val_kwargs.temperature=0.4)
)

# Проверяем, что LoRA загружен
print(f"\nLoRA загружен: {agent.use_lora}")
print(f"LoRA имя: {agent.lora_name}")


Проверка чекпоинта: /home/n.sorokin/verl-agent/checkpoints/verl_agent_craftext/gigpo_qwen2.5_1.5b_lora_run_20251110-160709/global_step_500
  Чекпоинт существует: True
  LoRA путь: /home/n.sorokin/verl-agent/checkpoints/verl_agent_craftext/gigpo_qwen2.5_1.5b_lora_run_20251110-160709/global_step_500/actor/lora_adapter
  LoRA существует: True
  LoRA файлы: ['adapter_config.json', 'adapter_model.safetensors']
Инициализация vLLM engine...
Включена поддержка LoRA (rank=64)
INFO 11-26 15:27:19 [config.py:717] This model supports multiple tasks: {'reward', 'generate', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
WARNING 11-26 15:27:19 [cuda.py:93] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-26 15:27:19 [llm_engine.py:240] Initializing a V0 LLM engine (v0.8.5) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.23it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.22it/s]


INFO 11-26 15:27:25 [loader.py:458] Loading weights took 0.87 seconds
INFO 11-26 15:27:25 [punica_selector.py:18] Using PunicaWrapperGPU.


INFO 11-26 15:27:25 [model_runner.py:1140] Model loading took 3.0567 GiB and 2.107120 seconds
INFO 11-26 15:27:28 [worker.py:287] Memory profiling takes 2.45 seconds
INFO 11-26 15:27:28 [worker.py:287] the current vLLM instance can use total_gpu_memory (79.26GiB) x gpu_memory_utilization (0.90) = 71.33GiB
INFO 11-26 15:27:28 [worker.py:287] model weights take 3.06GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.39GiB; the rest of the memory reserved for KV Cache is 66.79GiB.
INFO 11-26 15:27:28 [executor_base.py:112] # cuda blocks: 156323, # CPU blocks: 9362
INFO 11-26 15:27:28 [executor_base.py:117] Maximum concurrency for 2560 tokens per request: 977.02x
INFO 11-26 15:27:32 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 6.37 seconds
LoRA адаптер загружен из /home/n.sorokin/verl-agent/checkpoints/verl_agent_craftext/gigpo_qwen2.5_1.5b_lora_run_20251110-160709/global_step_500/actor/lora_adapter
SimpleAgent инициализирован!

LoRA

In [4]:
# Создаем простую среду (один env)
from agent_system.environments import make_envs
from omegaconf import OmegaConf, open_dict

# Создаем минимальный конфиг для среды
simple_cfg = OmegaConf.create({
    "env": {
        "env_name": "craftext/CraftextEnv",
        "craftext_settings": "achievements_collect_wood",
        "seed": 42,
        "rollout": {"n": 1},
        "max_steps": 100,
        "resources_per_worker": {  # Ресурсы для каждого env worker (обязательный параметр для make_envs)
            "num_cpus": 0.1,
            "num_gpus": 0,
        },
        "history_length": 1,
    },
    "data": {
        "val_batch_size": 1,
        "train_batch_size": 1,
    },
})

_, val_envs = make_envs(simple_cfg)

Loading textures from cache.
Textures successfully loaded from cache.


/home/n.sorokin/miniconda3/envs/verl-agent/lib/python3.12/subprocess.py:1883: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(
/home/n.sorokin/miniconda3/envs/verl-agent/lib/python3.12/subprocess.py:1883: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(
2025-11-26 15:27:40,698	INFO worker.py:1908 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


(pid=2283823) Loading textures from cache.
(pid=2283823) Textures successfully loaded from cache.


(CraftextWorker pid=2283823) Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
(CraftextWorker pid=2283823) Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
(CraftextWorker pid=2283823) Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
(CraftextWorker pid=2283823) See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


(CraftextWorker pid=2283823) _NamespacePath(['/home/n.sorokin/verl-agent/agent_system/environments/env_package/craftext/CrafText-super_igor_env_build/craftext/dataset'])
(CraftextWorker pid=2283823) /home/n.sorokin/verl-agent/agent_system/environments/env_package/craftext/CrafText-super_igor_env_build/craftext/dataset
(CraftextWorker pid=2283823) Initial number of instructions: 1
(CraftextWorker pid=2283823) Final number of instructions: 1
Craftext Training with 1 scenarios.


1it [00:00, 22075.28it/s]2283823) 
(CraftextWorker pid=2283823) 2025-11-26 15:27:51,781 - INFO - Initial number of instructions: 1
100%|██████████| 1/1 [00:00<00:00, 13.31it/s]
(CraftextWorker pid=2283823) 2025-11-26 15:27:51,877 - INFO - Final number of instructions: 1


(CraftextWorker pid=2283823) Initialized Instruction Wrapper with environment key: 0
(CraftextWorker pid=2283823) Def instr shape: (768,)
(pid=2283824) Loading textures from cache.
(pid=2283824) Textures successfully loaded from cache.


(CraftextWorker pid=2283824) Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
(CraftextWorker pid=2283824) Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
(CraftextWorker pid=2283824) Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
(CraftextWorker pid=2283824) See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


(CraftextWorker pid=2283824) _NamespacePath(['/home/n.sorokin/verl-agent/agent_system/environments/env_package/craftext/CrafText-super_igor_env_build/craftext/dataset'])
(CraftextWorker pid=2283824) /home/n.sorokin/verl-agent/agent_system/environments/env_package/craftext/CrafText-super_igor_env_build/craftext/dataset
(CraftextWorker pid=2283824) Initial number of instructions: 1
(CraftextWorker pid=2283824) Final number of instructions: 1
Craftext Evaluating with 1 scenarios.


1it [00:00, 22550.02it/s]2283824) 
(CraftextWorker pid=2283824) 2025-11-26 15:28:02,166 - INFO - Initial number of instructions: 1
100%|██████████| 1/1 [00:00<00:00, 13.47it/s]
(CraftextWorker pid=2283824) 2025-11-26 15:28:02,261 - INFO - Final number of instructions: 1


In [15]:
# Простой gym-подобный цикл
from agent_system.environments.env_package.craftext.projection import craftext_projection

obs, info = val_envs.reset({})
# print("Начальное наблюдение:")
# print(obs['text'][0][:500])  # первые 500 символов
# print("\n" + "="*50 + "\n")

step = 0
done = False
total_reward = 0.0
episode_reward = 0.0  # Для накопления награды эпизода (как в обучении)

while not done and step < 20:  # максимум 100 шагов
    # Получаем действие от агента
    action_text = agent.predict(obs['text'][0])
    print(f"Шаг {step}:")
    print(f"Действие: {action_text[:200]}")  # первые 200 символов
    
    # Проверяем валидность действия
    action_ids, valids = craftext_projection([action_text])
    is_valid = bool(valids[0])
    action_id = action_ids[0]
    # print(f"Action ID: {action_id}, Valid: {is_valid}")
    
    # Выполняем действие в среде
    next_obs, rewards, dones, infos = val_envs.step([action_text])
    
    reward = float(rewards[0])
    done = bool(dones[0])
    total_reward += reward
    episode_reward += reward  # Накопление награды эпизода
    
    # Проверяем is_action_valid из info
    action_valid_from_info = infos[0].get('is_action_valid', None)
    #print(f"Награда шага: {reward:.4f}, Награда эпизода (накопленная): {episode_reward:.4f}, Done: {done}")
    #print(f"Action valid (from info): {action_valid_from_info}")
    
    if step < 3:  # Показываем первые несколько наблюдений полностью
        # print(f"Новое наблюдение: {next_obs['text'][0][:500]}...")
        pass
    # print("\n" + "-"*50 + "\n")
    
    obs = next_obs
    step += 1

print(f"Итого: {step} шагов")
print(f"Общая награда (сумма шагов): {total_reward:.4f}")


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.19it/s, est. speed input: 1167.63 toks/s, output: 17.52 toks/s]


Шаг 0:
Действие: <action>up</action>


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.25it/s, est. speed input: 2193.57 toks/s, output: 18.00 toks/s]


Шаг 1:
Действие: <action>left</action>


...

Шаг 12:
Действие: <action>left</action>


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.21it/s, est. speed input: 2167.76 toks/s, output: 17.71 toks/s]


Шаг 13:
Действие: <action>left</action>


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s, est. speed input: 2066.50 toks/s, output: 16.89 toks/s]

Шаг 14:
Действие: <action>do</action>
Итого: 15 шагов
Общая награда (сумма шагов): 1.0200
